In [21]:
import random
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27018/")
db = client["yelp"]
collection = db["business"]

data = list(collection.find({}, {"_id": 0, "business_id": 0, "name": 0, "address": 0}).limit(1000))

# split into train and eval
random.shuffle(data)
split_idx = int(0.8 * len(data))
train_data = data[:split_idx]
eval_data = data[split_idx:]

print(f"train examples: {train_data[:5]}")

train examples: [{'city': 'Philadelphia', 'state': 'PA', 'postal_code': '19019', 'latitude': 40.1197128, 'longitude': -75.0097103, 'stars': 3.0, 'review_count': 8, 'is_open': 1, 'attributes': {'BusinessAcceptsCreditCards': True}, 'categories': ['Local Flavor', 'Team Building Activities', 'Active Life', 'Arts & Entertainment', 'Event Planning & Services', 'Walking Tours', 'Scavenger Hunts', 'Museums', 'Hotels & Travel', 'Tours'], 'hours': {'Monday': '9:0-17:0', 'Tuesday': '9:0-17:0', 'Wednesday': '9:0-17:0', 'Thursday': '9:0-17:0', 'Friday': '9:0-17:0'}}, {'city': 'Tucson', 'state': 'AZ', 'postal_code': '85704', 'latitude': 32.303741804, 'longitude': -111.009323614, 'stars': 4.5, 'review_count': 29, 'is_open': 1, 'attributes': {'BusinessAcceptsCreditCards': True, 'WiFi': 'free', 'WheelchairAccessible': True, 'RestaurantsPriceRange2': '2'}, 'categories': ['Party & Event Planning', 'Event Planning & Services', 'Venues & Event Spaces', 'Hotels & Travel', 'Hotels'], 'hours': {'Monday': '0:0

In [22]:
from origami import ModelConfig, OrigamiConfig, OrigamiPipeline, TrainingConfig
from origami.config import DataConfig
from origami.training import TableLogCallback, accuracy

config = OrigamiConfig(
    data=DataConfig(
        infer_schema=True,
        numeric_mode="scale",
        cat_threshold=5,
        n_bins=5,
    ),
    model=ModelConfig(
        backbone="transformer",
        kvpe_pooling="sum",
        d_model=32,
        n_heads=4,
        n_layers=6,
        d_ff=256,
        dropout=0.0,
    ),
    training=TrainingConfig(
        shuffle_keys=False,
        batch_size=16,
        warmup_steps=100,
        learning_rate=1e-3,
        eval_strategy="epoch",
        eval_steps=100,
        eval_metrics={"acc": accuracy},
        eval_sample_size=100,
        eval_on_train=True,
        target_key="stars",
        target_loss_weight=1.0,
        constrain_grammar=True,
        constrain_schema=True,
    ),
    device="mps",
)

pipeline = OrigamiPipeline(config)
pipeline.fit(
    train_data,
    eval_data=eval_data,
    callbacks=[TableLogCallback(print_every=10)],
    epochs=500,
    verbose=True,
)


Scaled fields (4):
  - latitude: mean=36.84, std=5.915
  - longitude: mean=-89.6, std=15.21
  - review_count: mean=49.21, std=135.1
  - stars: mean=3.634, std=0.9629
Vocabulary size: 1587
Derived schema:
{
  "type": "object",
  "properties": {
    "attributes": {
      "properties": {
        "AcceptsInsurance": {
          "type": "boolean",
          "enum": [
            false,
            true
          ]
        },
        "Alcohol": {
          "type": "string",
          "enum": [
            "beer_and_wine",
            "full_bar",
            "none"
          ]
        },
        "Ambience": {
          "properties": {
            "casual": {
              "type": [
                "boolean",
                "null"
              ],
              "enum": [
                null,
                false,
                true
              ]
            },
            "classy": {
              "type": [
                "boolean",
                "null"
              ],
             

OrigamiPipeline(numeric_mode='scale', fitted)

In [ ]:
import json


gen = pipeline.generate(100)

for i, sample in enumerate(gen):
    print(f"Sample {i}\n{json.dumps(sample, indent=2)}\n")

Sample 0: {'city': 'St. Louis', 'state': 'NV', 'postal_code': '93101', 'latitude': 40.0520643, 'longitude': -119.510772, 'stars': 3.5, 'review_count': 26, 'is_open': 1, 'attributes': {'WiFi': 'free', 'RestaurantsPriceRange2': '2', 'Alcohol': 'beer_and_wine', 'Caters': False, 'BusinessAcceptsCreditCards': 1, 'BusinessParking': {'garage': False, 'street': 1, 'validated': False, 'lot': 1, 'valet': False}, 'HasTV': 1, 'OutdoorSeating': 1}, 'categories': ['Home Services', 'Restaurants', 'Cajun/Creole', 'Party & Event Planning', 'Florists', 'Urgent Care', 'Shanghainese', 'Restaurants', 'American (New)', 'Historical Tours', 'Steakhouses', 'Cafes', 'Sandwiches'], 'hours': {'Monday': '9:0-16:0', 'Tuesday': '7:30-17:30', 'Wednesday': '8:0-23:0', 'Thursday': '8:0-17:0', 'Friday': '6:30-17:30', 'Saturday': '9:0-13:0', 'Sunday': '10:0-21:0'}}
Sample 1: {'city': 'Edmonton', 'state': 'PA', 'postal_code': '33617', 'stars': 3.5, 'review_count': 37, 'is_open': 1, 'attributes': {'BikeParking': False, 'Bu